### Importação das bibliotecas

In [1]:
import cv2 as cv
from ultralytics import YOLO
import random

### Configurações

In [10]:
MODEL_SOURCE_PATH = 'yolov8n.pt'
VIDEO_SOURCE_PATH = 'files/vehicle-counting.mp4' # WEB CAM = 0

# Definindo a largura e a altura dos frames
LARGURA_FRAME = 640
ALTURA_FRAME = 480

LIMIAR_CONFIANCA = 0.3 # Limiar de confiança

# Se SKIP_FRAMES for 2, a cada 2 frames será processado
SKIP_FRAMES = 2
frame_count = 0

### Modelo pré-treinado

In [3]:
# Carregando o modelo pré-treinado YOLOv8n
model = YOLO(MODEL_SOURCE_PATH, "v8")

# Obtendo o nome de todas as classes do modelo
lista_classes = list(model.model.names.values())

# Obtendo o número máximo de classes detectadas pelo modelo
num_classes = len(model.model.names)

# Vamos gerar cores aleatórias para as classes
cores_deteccao = []
for i in range(num_classes):
    r = random.randint(0, 255)
    g = random.randint(0, 255)
    b = random.randint(0, 255)
    cores_deteccao.append((b, g, r))

In [4]:
lista_classes

['person',
 'bicycle',
 'car',
 'motorcycle',
 'airplane',
 'bus',
 'train',
 'truck',
 'boat',
 'traffic light',
 'fire hydrant',
 'stop sign',
 'parking meter',
 'bench',
 'bird',
 'cat',
 'dog',
 'horse',
 'sheep',
 'cow',
 'elephant',
 'bear',
 'zebra',
 'giraffe',
 'backpack',
 'umbrella',
 'handbag',
 'tie',
 'suitcase',
 'frisbee',
 'skis',
 'snowboard',
 'sports ball',
 'kite',
 'baseball bat',
 'baseball glove',
 'skateboard',
 'surfboard',
 'tennis racket',
 'bottle',
 'wine glass',
 'cup',
 'fork',
 'knife',
 'spoon',
 'bowl',
 'banana',
 'apple',
 'sandwich',
 'orange',
 'broccoli',
 'carrot',
 'hot dog',
 'pizza',
 'donut',
 'cake',
 'chair',
 'couch',
 'potted plant',
 'bed',
 'dining table',
 'toilet',
 'tv',
 'laptop',
 'mouse',
 'remote',
 'keyboard',
 'cell phone',
 'microwave',
 'oven',
 'toaster',
 'sink',
 'refrigerator',
 'book',
 'clock',
 'vase',
 'scissors',
 'teddy bear',
 'hair drier',
 'toothbrush']

In [11]:
def process_video(source_path: str | int = 0) -> None:
    global frame_count

    # Carregando o vídeo
    cap = cv.VideoCapture(source_path)

    while cap.isOpened():
        # Capturando frame a frame
        ret, frame = cap.read()

        if not ret:
            print("FIM!")
            break

        frame_count += 1
        if frame_count % SKIP_FRAMES != 0:
            continue

        # Redimensionando o frame
        frame = cv.resize(frame, (LARGURA_FRAME, ALTURA_FRAME))

        # Realizando a detecção de objetos no frame
        deteccoes = model.track(source=[frame], conf=LIMIAR_CONFIANCA, save=False, iou=0.70, imgsz=640)

        # Convertendo a saída do modelo para um numpy array
        if len(deteccoes) != 0:
            for deteccao in deteccoes:
                caixas = deteccao.boxes
                for caixa in caixas:
                    id_classe = int(caixa.cls[0])
                    confianca = float(caixa.conf[0])
                    bb = caixa.xyxy[0]

                    # Desenhando uma caixa delimitadora ao redor do objeto detectado
                    cv.rectangle(frame,
                                (int(bb[0]), int(bb[1])),
                                (int(bb[2]), int(bb[3])),
                                cores_deteccao[id_classe],
                                3)
                    
                    # Exibindo o nome da classe e a confiança da detecção
                    fonte = cv.FONT_HERSHEY_COMPLEX
                    cv.putText(
                        frame,
                        lista_classes[int(id_classe)]
                        + " "
                        + str(round(confianca, 3))
                        + "%",
                        (int(bb[0]), int(bb[1]) - 10),
                        fonte,
                        1,
                        (255, 255, 255),
                        2,
                    )

        # Exibindo o frame resultante
        cv.imshow('Detecção de Objetos', frame)

        # Terminando a execução quando "Q" é pressionado
        if cv.waitKey(1) == ord('q'):
            break

    cap.release() # Libera a captura de vídeo
    cv.destroyAllWindows() # Fecha todas as janelas

In [8]:
# Executar web cam
process_video()


0: 480x640 (no detections), 251.9ms
Speed: 15.0ms preprocess, 251.9ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 184.4ms
Speed: 1.4ms preprocess, 184.4ms inference, 2.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 228.3ms
Speed: 0.0ms preprocess, 228.3ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 181.0ms
Speed: 12.0ms preprocess, 181.0ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 163.6ms
Speed: 0.0ms preprocess, 163.6ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 195.1ms
Speed: 4.2ms preprocess, 195.1ms inference, 0.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 194.1ms
Speed: 3.2ms preprocess, 194.1ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 164.0ms
Speed: 2.2ms pre

In [12]:
# Executar video de demonstração
process_video(source_path=VIDEO_SOURCE_PATH)


0: 480x640 2 cars, 243.5ms
Speed: 7.6ms preprocess, 243.5ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 3 cars, 1 bus, 267.2ms
Speed: 2.0ms preprocess, 267.2ms inference, 15.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 cars, 263.4ms
Speed: 3.0ms preprocess, 263.4ms inference, 4.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 cars, 1 truck, 241.8ms
Speed: 4.0ms preprocess, 241.8ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 cars, 1 truck, 167.4ms
Speed: 2.0ms preprocess, 167.4ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 cars, 178.3ms
Speed: 0.0ms preprocess, 178.3ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 cars, 181.1ms
Speed: 0.0ms preprocess, 181.1ms inference, 1.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 cars, 222.5ms
Speed: 1.7ms preprocess, 222.5ms inference, 0.0ms postprocess pe